# lwm1 v2
<br>모델 크기 차이에 따른 성능 차이 비교
<br> element_length=64, d_model=128
<br> in_proj: F_in(2048) → 512 → 256 → 64
<br> 예측 범위를 16(observation) -> 1(target) 이 아닌 16->4로 진행

# Import

In [22]:
import os
import sys

import numpy as  np
from lwm_model1 import lwm  # 클래스 import
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, Subset
import torch.nn as nn   
import time

from PIL import Image
import numpy as np

from deepverse import ParameterManager
from deepverse.scenario import ScenarioManager
from deepverse import Dataset

from deepverse.visualizers import ImageVisualizer, LidarVisualizer

# Setting 
### CUDA 몇번 사용하는지 잘 확인하기

In [23]:
# Scenes 1000
## Subcarriers 64

scenarios_name = "DT31"
config_path = f"scenarios/{scenarios_name}/param/config.m"
param_manager = ParameterManager(config_path)

params = param_manager.get_params()

param_manager.params["scenes"] =list(range(100))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(64))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")


현재 사용 중인 장치: cuda:0


# Generate a dataset

In [24]:
# scenes / subcarriers는 그대로
param_manager.params["radar"]["enable"] = False

# lidar가 dict로 존재하면 disable
if isinstance(param_manager.params.get("lidar", None), dict):
    param_manager.params["lidar"]["enable"] = False

# position이 dict로 존재하면 disable
if isinstance(param_manager.params.get("position", None), dict):
    param_manager.params["position"]["enable"] = False

# Radar, LiDAR, Position 사용하지 않으므로 False 
dataset = Dataset(param_manager)

Generating camera dataset: ⏳ In progress
Generating camera dataset: ✅ Completed (0.00s)
Generating LiDAR dataset: ⏳ In progress
Generating LiDAR dataset: ✅ Completed (0.00s)
Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (0.83s)


# Prerocessing

#### Channel

In [25]:
def get_coeffs_from_frame(frame, ue_idx=0):
    ue_obj = frame["ue"]

    # 케이스1) list/tuple이면 ue_idx로 선택
    if isinstance(ue_obj, (list, tuple)):
        ch_obj = ue_obj[ue_idx]
    else:
        # 케이스2) 단일 OFDMChannel이면 그대로 사용
        ch_obj = ue_obj

    # coeffs는 dict key가 아니라 attribute일 확률이 매우 큼
    if hasattr(ch_obj, "coeffs"):
        return ch_obj.coeffs

    # 혹시 dict라면 마지막 보험
    if isinstance(ch_obj, dict) and "coeffs" in ch_obj:
        return ch_obj["coeffs"]

    raise TypeError(f"Cannot get coeffs. ue type={type(ue_obj)}, ch type={type(ch_obj)}")


In [26]:
def get_train_min_max_realimag(frames, train_idx, us_idx=0):

    rmin, rmax =  float('inf'), float('-inf')
    imin, imax =  float('inf'), float('-inf')

    print("Calculating min/max over training set...")

    for t  in train_idx:
        frame  = frames[t]
        cooeffs  = get_coeffs_from_frame(frame, us_idx)  # (N_subcarriers, )

        rmin = min(rmin, float(cooeffs.real.min()))
        rmax = max(rmax, float(cooeffs.real.max()))
        imin = min(imin, float(cooeffs.imag.min()))
        imax = max(imax, float(cooeffs.imag.max()))

    print(f"Done. rmin={rmin}, rmax={rmax}, imin={imin}, imax={imax}")
    return (rmin, rmax), (imin, imax)

In [27]:
def preprocess_channel_coeffs_minmax(coeffs_np, r_min, r_max, i_min, i_max, device=device, eps=1e-16):
    # Convert Numpy to Tensor
    coeffs = torch.from_numpy(coeffs_np).to(torch.complex64)
    
    
    r = coeffs.real
    i = coeffs.imag
    
    # Min-Max Scaling [0, 1]
    # Add eps to denominator to prevent division by zero
    r_scaled = (r - r_min) / max(r_max - r_min, eps)
    i_scaled = (i - i_min) / max(i_max - i_min, eps)
    
    # Concat (Maintains shape like (..., 2*subcarriers))
    H = torch.cat([r_scaled, i_scaled], dim=-1).to(device, dtype=torch.float32)
    return H

#### image

# Dataset 구현
<br> 예측이 16->4로 바뀌면 아래 코드도 바뀌어야함

In [28]:
def flatten_comm_frames(comm):
    frames = []
    for row in comm.data:
        for d in row:
            frames.append(d)
    return frames

class ChannelNextStepDatasetGPU(TorchDataset):
    def __init__(self, comm_frames, ue_idx=0, past_len=16, pre_len=4, device=device,
                 r_min=0.0, r_max=1.0, i_min=0.0, i_max=1.0):
        self.comm_frames = comm_frames
        self.ue_idx = ue_idx
        self.past_len = past_len
        # ✅ pre_len 추가 
        self.pre_len = pre_len 
        self.device = device

        self.r_min, self.r_max = r_min, r_max
        self.i_min, self.i_max = i_min, i_max

        self.N = len(self.comm_frames)
        self.valid_start = past_len - 1
        # ✅ t + pre_len 까지 필요
        self.valid_end = self.N - self.pre_len - 1

    def __len__(self):
        return self.valid_end - self.valid_start + 1

    def __getitem__(self, idx):
        t = self.valid_start + idx

        # 1) Channel Past: (T, 2048)
        ch_list = []
        for k in range(t - self.past_len + 1, t + 1):
            coeffs_np = get_coeffs_from_frame(self.comm_frames[k], ue_idx=self.ue_idx)
            h = preprocess_channel_coeffs_minmax(
                coeffs_np,
                self.r_min, self.r_max, self.i_min, self.i_max,
                device=self.device
            ).reshape(-1)  # (2048,)
            ch_list.append(h)
        channel_past = torch.stack(ch_list, dim=0)  # (past_len, 2048)

        # 2) Target Next 4: flatten -> (4*2048,)
        # target이 1이 아닌 4인 점 주의 
        tgt_list = []
        for j in range(1, self.pre_len + 1):
            coeffs_np_next = get_coeffs_from_frame(self.comm_frames[t + j], ue_idx=self.ue_idx)
            hj = preprocess_channel_coeffs_minmax(
                coeffs_np_next,
                self.r_min, self.r_max, self.i_min, self.i_max,
                device=self.device
            ).reshape(-1)  # (2048,)
            tgt_list.append(hj)

        target = torch.stack(tgt_list, dim=0)  # (pre_len, 2048)

        return channel_past, target

# DataLoader 구현

In [29]:
comm_frames = flatten_comm_frames(dataset.comm_dataset)

ds = ChannelNextStepDatasetGPU(
    comm_frames=comm_frames,
    ue_idx=0,
    past_len=16,
    pre_len=4,
    device=device
)

loader = DataLoader(
    ds,
    batch_size=8,
    shuffle=True,
    num_workers=0,     
    pin_memory=False   
)
ch, y = next(iter(loader))
print(ch.shape, y.shape)
print(ch.device, y.device)


torch.Size([8, 16, 2048]) torch.Size([8, 4, 2048])
cuda:0 cuda:0


# Fine-tuning

In [30]:
import torch
import torch.nn as nn

class LWMForecasterNoEdit(nn.Module):
    """
    Input : ch  (B,T,F_in)
    Output: yhat(B,pre_len,F_step)
    """
    def __init__(
        self,
        base_lwm: nn.Module,
        F_in: int,
        pre_len: int,
        F_step: int,
        pool: str = "last",
        freeze_backbone: bool = False,
    ):
        super().__init__()
        self.base = base_lwm
        self.pool = pool
        self.pre_len = pre_len
        self.F_step = F_step

        n_dim   = self.base.embedding.element_length  # 예: 64
        d_model = self.base.embedding.d_model         # 예: 128

        self.in_proj = nn.Sequential(
            nn.Linear(F_in, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, n_dim),
        )

        # ✅ 출력은 pre_len * F_step 로 만들어야 (B,pre_len,F_step)로 reshape 가능
        self.head = nn.Linear(d_model, pre_len * F_step)

        if freeze_backbone:
            for p in self.base.parameters():
                p.requires_grad = False
            for p in self.in_proj.parameters():
                p.requires_grad = True
            for p in self.head.parameters():
                p.requires_grad = True

    def forward(self, ch):
        B = ch.size(0)

        x = self.in_proj(ch)          # (B,T,F_in) -> (B,T,n_dim)
        out = self.base.embedding(x)  # (B,T,d_model) 형태가 되도록 base.embedding 구현에 따름
        for layer in self.base.layers:
            out, _ = layer(out)

        z = out[:, -1, :] if self.pool == "last" else out.mean(dim=1)  # (B,d_model)
        yhat_flat = self.head(z)  # (B, pre_len*F_step)

        return yhat_flat.view(B, self.pre_len, self.F_step)  # (B,pre_len,F_step)

# NMSE(dB)

In [31]:
@torch.no_grad()
def nmse_db(yhat: torch.Tensor, y: torch.Tensor, eps: float = 1e-16) -> torch.Tensor:
    # yhat, y: (B, pre_len, F_step)
    num = torch.sum((yhat - y) ** 2).clamp_min(eps)  # batch 전체 합
    den = torch.sum(y ** 2).clamp_min(eps)
    nmse = num / den
    return 10.0 * torch.log10(nmse)

# Train/val Split

In [32]:
# 겹치는 부분이 있는지 확인하는 함수
# Target이 1일 때만 아래 코드 작성
# def used_frames_for_ds_indices(ds, ds_indices):
#     used = set()
#     for i in ds_indices:
#         t = ds.valid_start + i
#         # inputs use [t-T+1 .. t], target uses (t+1)
#         used.update(range(t - T + 1, t + 2))
#     return used

def used_frames_for_ds_indices(ds, ds_indices):
    used = set()
    T = ds.past_len
    P = ds.pre_len
    for i in ds_indices:
        t = ds.valid_start + i
        used.update(range(t - T + 1, t + P + 1))  # [t-15 .. t+4]
    return used

In [33]:
n_total = len(ds)
T = ds.past_len  # = past_len
P = ds.pre_len
# ---- choose k so that train=3k, val=k and NO frame-overlap is possible ----
k_max = (n_total - (T + P - 1)) // 4
if k_max <= 0:
    raise ValueError(
        f"Not enough data for strict non-overlap 3:1. "
        f"n_total={n_total}, past_len(T)={T}. "
        f"Try increasing scenes or decreasing past_len."
    )

k = k_max               # use as much data as possible under constraints
n_val = k
n_train = 3 * k

train_idx = list(range(0, n_train))
val_idx   = list(range(n_total - n_val, n_total))

print("n_total:", n_total, "T:", T)
print("train samples:", len(train_idx), "val samples:", len(val_idx), "ratio:", len(train_idx)/len(val_idx))

# ---- min/max ONLY from train (same as your original policy) ----

# (4,2048) 이 되므로 아래 min/max 계산도 바꿔서
# ---- min/max ONLY from train, but using ALL frames used by train samples ----
train_used_frames = sorted(used_frames_for_ds_indices(ds, train_idx))  # <-- 핵심

(real_min, real_max), (imag_min, imag_max) = get_train_min_max_realimag(
    comm_frames, train_used_frames, us_idx=0
)

ds.r_min = real_min
ds.r_max = real_max
ds.i_min = imag_min
ds.i_max = imag_max

print("Dataset statistical values set in the dataset (from all train-used frames).")

train_ds = Subset(ds, train_idx)
val_ds   = Subset(ds, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

F_in = 2048


overlap = used_frames_for_ds_indices(ds, train_idx).intersection(
          used_frames_for_ds_indices(ds, val_idx))
print("frame-overlap count:", len(overlap))
assert len(overlap) == 0, "Leakage detected: train/val share frames!"

# Verify (channel-only)
ch, y = next(iter(train_loader))
assert y.ndim == 3, f"Expected (B, pre_len, F_step), got {y.shape}"

pre_len = y.shape[1]
F_step  = y.shape[2]
F_out_total = pre_len * F_step

print("\n=== Data Check ===")
print(f"y stats | min: {y.min().item():.4f}, max: {y.max().item():.4f}")
print("If scaling worked correctly, values should be within [0, 1].")

# (선택) ch도 같이 범위 확인하고 싶으면
print(f"ch stats | min: {ch.min().item():.4f}, max: {ch.max().item():.4f}")

n_total: 81 T: 16
train samples: 45 val samples: 15 ratio: 3.0
Calculating min/max over training set...
Done. rmin=-1.6025904539973852e-06, rmax=1.6123052658255991e-06, imin=-1.6147482377856819e-06, imax=1.5844537584687652e-06
Dataset statistical values set in the dataset (from all train-used frames).
frame-overlap count: 0

=== Data Check ===
y stats | min: 0.1673, max: 0.8381
If scaling worked correctly, values should be within [0, 1].
ch stats | min: 0.0000, max: 1.0000


# Train/Eval 함수

In [34]:
# batch check
ch, y = next(iter(train_loader))
print(ch.shape, y.shape)

def train_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, y in loader:
        ch = ch.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        yhat = model(ch)                
        loss = F.mse_loss(yhat, y)
        loss.backward()

        if grad_clip and grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        total_loss += loss.item()
        total_nmse += nmse_db(yhat.detach(), y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, y in loader:
        ch = ch.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        yhat = model(ch)                 
        loss = F.mse_loss(yhat, y)

        total_loss += loss.item()
        total_nmse += nmse_db(yhat, y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)

torch.Size([32, 16, 2048]) torch.Size([32, 4, 2048])


# 모델 생성 및 확인

In [35]:
# 배치 하나로 F_in/F_out 자동 확정
ch, y = next(iter(train_loader))
F_in    = ch.shape[-1]      # 2048
pre_len = y.shape[1]        # 4
F_step  = y.shape[2]        # 2048
F_out_total = pre_len * F_step   # 8192

print("Detected:",
      "F_in=", F_in,
      "pre_len=", pre_len,
      "F_step=", F_step,
      "F_out_total=", F_out_total)
print("Batch devices:", ch.device, y.device)

backbone = lwm().to(device)
model = LWMForecasterNoEdit(
    base_lwm=backbone,
    F_in=F_in,
    pre_len=pre_len,
    F_step=F_step,
    pool="last",
    freeze_backbone=False,
).to(device)

# sanity forward
model.eval()
with torch.no_grad():
    # ds가 이미 cuda 텐서 반환이면 아래 .to(device) 생략 가능
    yhat = model(ch.to(device))
print("yhat:", yhat.shape, "y:", y.shape)



Detected: F_in= 2048 pre_len= 4 F_step= 2048 F_out_total= 8192
Batch devices: cuda:0 cuda:0
yhat: torch.Size([32, 4, 2048]) y: torch.Size([32, 4, 2048])


In [36]:
import inspect
from lwm_model1 import lwm
print(inspect.signature(lwm.__init__))

(self, element_length=64, d_model=128, max_len=129, n_layers=12)


# Optimizer/Scheduler 설정

In [37]:
# requires_grad=True인 파라미터만 학습
trainable_params = [p for p in model.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in trainable_params))

optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

# (선택) cosine scheduler
epochs = 500
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


trainable params: 4686080


# 학습 루프 및 checkpoint 저장

In [38]:
import time
import os
import torch

best_val = float("inf")
best_val_nmse = None
best_epoch = None

t_train0 = time.time()

log_path = "lwm_model1_scene100_v1.txt"

with open(log_path, "w", encoding="utf-8") as f:
    for epoch in range(1, epochs + 1):
        t0 = time.time()

        tr_loss, tr_nmse = train_one_epoch(
            model, train_loader, optimizer, device=device, grad_clip=1.0
        )
        va_loss, va_nmse = evaluate(model, val_loader, device=device)

        scheduler.step()
        dt = time.time() - t0

        line = (
            f"[{epoch:02d}/{epochs}] "
            f"train loss={tr_loss:.6f}, nmse(dB)={tr_nmse:.4f} | "
            f"val loss={va_loss:.6f}, nmse(dB)={va_nmse:.4f} | "
            f"{dt:.1f}s"
        )
        print(line)
        f.write(line + "\n")
        f.flush()

        if va_loss < best_val:
            best_val = va_loss
            best_val_nmse = va_nmse
            best_epoch = epoch

            ckpt_path = "bestc_channel_only_finetune.pt"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),

                    "F_in": F_in,
                    "pre_len": pre_len,
                    "F_step": F_step,
                    "F_out_total": pre_len * F_step,

                    "best_val_loss": best_val,
                    "best_val_nmse_db": best_val_nmse,
                },
                ckpt_path,
            )

            save_line = (
                f"  ↳ saved {ckpt_path} | best@{best_epoch}: "
                f"val loss={best_val:.6f}, val nmse(dB)={best_val_nmse:.4f}"
            )
            print(save_line)
            f.write(save_line + "\n")
            f.flush()

total_sec = time.time() - t_train0
h = int(total_sec // 3600)
m = int((total_sec % 3600) // 60)
s = total_sec % 60

summary1 = "\n=== Training Summary ==="
summary2 = f"Total time: {total_sec:.1f}s ({h}h {m}m {s:.1f}s)"
summary3 = (
    f"Best epoch: {best_epoch} | best val loss={best_val:.6f}, "
    f"best val nmse(dB)={best_val_nmse:.4f}"
)

print(summary1)
print(summary2)
print(summary3)

with open(log_path, "a", encoding="utf-8") as f:
    f.write(summary1 + "\n")
    f.write(summary2 + "\n")
    f.write(summary3 + "\n")

print("Log saved to:", os.path.abspath(log_path))

[01/500] train loss=0.599132, nmse(dB)=3.4493 | val loss=0.540924, nmse(dB)=3.2496 | 0.1s


  ↳ saved bestc_channel_only_finetune.pt | best@1: val loss=0.540924, val nmse(dB)=3.2496
[02/500] train loss=0.552099, nmse(dB)=3.0832 | val loss=0.507476, nmse(dB)=2.9724 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@2: val loss=0.507476, val nmse(dB)=2.9724
[03/500] train loss=0.522184, nmse(dB)=2.8455 | val loss=0.479900, nmse(dB)=2.7298 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@3: val loss=0.479900, val nmse(dB)=2.7298
[04/500] train loss=0.496598, nmse(dB)=2.6156 | val loss=0.455779, nmse(dB)=2.5058 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@4: val loss=0.455779, val nmse(dB)=2.5058
[05/500] train loss=0.470245, nmse(dB)=2.4445 | val loss=0.434056, nmse(dB)=2.2937 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@5: val loss=0.434056, val nmse(dB)=2.2937
[06/500] train loss=0.452735, nmse(dB)=2.2165 | val loss=0.413707, nmse(dB)=2.0852 | 0.1s
  ↳ saved bestc_channel_only_finetune.pt | best@6: val loss=0.413707, val nmse(dB)=2.0852
[07/500] t

# 런타임 체크

In [39]:
ch, y = next(iter(train_loader))
print("y abs mean:", y.abs().mean().item())
print("y abs max :", y.abs().max().item())
print("y power   :", (y**2).mean().item())

with torch.no_grad():
    yhat = model(ch.to(device))   
print("yhat abs mean:", yhat.abs().mean().item())
print("yhat abs max :", yhat.abs().max().item())
print("yhat power   :", (yhat**2).mean().item())

y abs mean: 0.4969414174556732
y abs max : 0.8380954265594482
y power   : 0.27047955989837646
yhat abs mean: 0.4991978406906128
yhat abs max : 0.5690182447433472
yhat power   : 0.24958902597427368


In [40]:
y

tensor([[[0.4064, 0.4125, 0.4173,  ..., 0.6007, 0.5946, 0.5867],
         [0.4694, 0.4760, 0.4836,  ..., 0.4542, 0.4428, 0.4308],
         [0.3565, 0.3550, 0.3542,  ..., 0.6187, 0.6105, 0.6004],
         [0.3907, 0.3841, 0.3781,  ..., 0.6456, 0.6340, 0.6196]],

        [[0.5981, 0.6071, 0.6164,  ..., 0.6427, 0.6380, 0.6323],
         [0.2726, 0.2719, 0.2713,  ..., 0.7109, 0.7131, 0.7159],
         [0.6410, 0.6336, 0.6262,  ..., 0.3531, 0.3630, 0.3737],
         [0.6166, 0.6213, 0.6243,  ..., 0.2880, 0.2802, 0.2737]],

        [[0.5586, 0.5688, 0.5797,  ..., 0.6582, 0.6710, 0.6817],
         [0.6315, 0.6318, 0.6311,  ..., 0.5701, 0.5622, 0.5518],
         [0.5919, 0.5841, 0.5780,  ..., 0.4748, 0.4650, 0.4556],
         [0.3959, 0.4081, 0.4230,  ..., 0.6623, 0.6670, 0.6714]],

        ...,

        [[0.7099, 0.7141, 0.7180,  ..., 0.7404, 0.7373, 0.7336],
         [0.5981, 0.6071, 0.6164,  ..., 0.6427, 0.6380, 0.6323],
         [0.2726, 0.2719, 0.2713,  ..., 0.7109, 0.7131, 0.7159],
     

In [41]:
yhat

tensor([[[0.5531, 0.5484, 0.5482,  ..., 0.5158, 0.5121, 0.5183],
         [0.5564, 0.5592, 0.5622,  ..., 0.5053, 0.5060, 0.5075],
         [0.5457, 0.5464, 0.5509,  ..., 0.5158, 0.5142, 0.5130],
         [0.5341, 0.5425, 0.5406,  ..., 0.5227, 0.5173, 0.5197]],

        [[0.5524, 0.5489, 0.5487,  ..., 0.5156, 0.5126, 0.5188],
         [0.5558, 0.5606, 0.5625,  ..., 0.5055, 0.5057, 0.5063],
         [0.5461, 0.5465, 0.5510,  ..., 0.5163, 0.5134, 0.5133],
         [0.5347, 0.5422, 0.5415,  ..., 0.5212, 0.5181, 0.5207]],

        [[0.5531, 0.5483, 0.5485,  ..., 0.5158, 0.5127, 0.5184],
         [0.5563, 0.5596, 0.5624,  ..., 0.5055, 0.5062, 0.5074],
         [0.5461, 0.5464, 0.5508,  ..., 0.5157, 0.5142, 0.5130],
         [0.5341, 0.5423, 0.5406,  ..., 0.5227, 0.5174, 0.5200]],

        ...,

        [[0.5522, 0.5484, 0.5496,  ..., 0.5158, 0.5132, 0.5190],
         [0.5545, 0.5599, 0.5625,  ..., 0.5051, 0.5051, 0.5060],
         [0.5474, 0.5462, 0.5518,  ..., 0.5167, 0.5129, 0.5137],
     

# 데이터 입력 형태

In [42]:
print("=== dataset sizes ===")
print("N(comm_frames):", len(comm_frames))
print("past_len      :", ds.past_len)
print("pre_len       :", ds.pre_len)
print("len(ds)       :", len(ds))
print("len(train_ds) :", len(train_ds))
print("len(val_ds)   :", len(val_ds))
print("len(train_loader):", len(train_loader))
print("len(val_loader)  :", len(val_loader))

print("\n=== one batch shapes ===")
ch, y = next(iter(train_loader))

print("ch :", tuple(ch.shape), " -> (B, T, F_in)")
print("y  :", tuple(y.shape),  " -> (B, pre_len, F_step)")

with torch.no_grad():
    yhat = model(ch.to(device))

print("yhat:", tuple(yhat.shape), " -> (B, pre_len, F_step)")

# ✅ sanity: shape 일치 확인
assert yhat.shape == y.shape, f"shape mismatch: yhat={yhat.shape}, y={y.shape}"

B, P, F = yhat.shape
print("this forward predicted samples:", B, "(=B)")
print("each sample predicts steps    :", P, "(=pre_len)")
print("each step predicts elements   :", F, "(=F_step)")

# (선택) 첫 샘플의 1-step 일부 값 확인
print("\n=== sample preview ===")
print("y[0,0,:10]   :", y[0,0,:10].detach().cpu())
print("yhat[0,0,:10]:", yhat[0,0,:10].detach().cpu())

=== dataset sizes ===
N(comm_frames): 100
past_len      : 16
pre_len       : 4
len(ds)       : 81
len(train_ds) : 45
len(val_ds)   : 15
len(train_loader): 2
len(val_loader)  : 1

=== one batch shapes ===
ch : (32, 16, 2048)  -> (B, T, F_in)
y  : (32, 4, 2048)  -> (B, pre_len, F_step)
yhat: (32, 4, 2048)  -> (B, pre_len, F_step)
this forward predicted samples: 32 (=B)
each sample predicts steps    : 4 (=pre_len)
each step predicts elements   : 2048 (=F_step)

=== sample preview ===
y[0,0,:10]   : tensor([0.5586, 0.5688, 0.5797, 0.5902, 0.5994, 0.6073, 0.6141, 0.6206, 0.6277,
        0.6354])
yhat[0,0,:10]: tensor([0.5531, 0.5483, 0.5485, 0.5586, 0.5552, 0.5558, 0.5472, 0.5462, 0.5511,
        0.5491])
